# Trabajo Práctico: Impacto de la IA en el Empleo (2020-2026)
## Sector Asignado: Finanzas (Finance)

### 1. Introducción Académica
El avance de la Inteligencia Artificial no es solo un fenómeno técnico, sino una fuerza socioeconómica que está redefiniendo la naturaleza del trabajo. En los próximos años, diversos sectores experimentarán cambios sin precedentes en sus estructuras salariales, demandas de habilidades y estabilidad laboral. Este Trabajo Práctico propone analizar estas tendencias con rigor científico y pensamiento crítico.

### 2. Ficha del Proyecto: Grupo 2
*   **Sector**: Finanzas.
*   **Objetivo General**: Analizar el impacto de la IA en la volatilidad salarial y la brecha de habilidades dentro del sector financiero, identificando países con mayor estrés de transición.

#### Setup y parametros

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

csv_base_file_path = '../MaterialBase/ai_job_replacement_dirty.csv'

### 3. Fase I: Diagnóstico y Saneamiento (Caja Negra)
Ustedes trabajarán con la fuente de datos `ai_job_replacement_dirty.csv`. **Advertencia**: La fuente posee anomalías estructuradas.
- **Auditoría de Calidad**: Realicen un reporte inicial sobre el estado de la información. Deben identificar inconsistencias lógicas, errores de formato y vacíos de datos sin guía externa.
- **Protocolo de Curación**: Implementen y documenten un proceso sistemático de limpieza. Cada decisión (casteo de tipos, manejo de nulos, eliminación de duplicados) debe estar técnicamente justificada en su bitácora.
- **Normalización**: Traduzcan y estandaricen los campos para asegurar una lectura fluida en español.

#### Carga y muestra rápida 

In [ ]:
df = pd.read_csv(csv_base_file_path)

print("Info del DataFrame:")
display(df.info())

print("Primeras filas del DataFrame:")
display(df.head())
print("Últimas filas del DataFrame:")
display(df.tail())
print("Muestra aleatoria de 10 filas:")
display(df.sample(n=10))

#### Analisis de vacios y filas duplicadas

In [ ]:
print("Conteo de valores nulos por columna:")
display(df.isnull().sum())

print("Número de filas repetidas:", df.duplicated().sum())

#### Identificación de inconsistencias en valores categoricos

In [ ]:
object_columns = df.select_dtypes(include=['object']).columns

for col in object_columns:
    print(f"Distribución de categorias para {col}:")
    display(df[col].value_counts(normalize=True, sort=False).sort_index())

#### Limpieza inicial del dataset

*Puntos a atacar:*

- Repetidos:
  - Detectados: 
    - 50 filas repetidas
  - Estrategia:
    - Se dejara una fila única por cada repetido
- Conflictos en columnas categóricas:
  - Detectados:
    - Industry presenta inconsistencias en su uso de mayúsculas y minúsculas
    - Salary_before_usd y salary_after_usd: floats que estan en formato string y presentan uso inconsistente del carácter "$"
  - Estrategia:
    - Industry sera pasado a solo minúscula de manera general
    - Salary_before_usd y salary_after_usd seran convertidos a float
- Vacios:
  - Detectados:
    - salary_after_usd: 754
    - salary_before_usd: 753
    - automation_risk_percent: 752
    - industry: 751
  - Estrategia:
    - Los casos vacios de Industry se completaran con "unknown"
    - Los vacios de automation_risk_percent seran completados analizando el promedio a nivel job_role/industry.
    - Los vacios de salary_before_usd y salary_after_usd seran completados analizando la mediana a nivel job_role/industry.
      - Se usa la mediana ya que los sueldos muy altos o muy bajos harian del promedio un valor poco representativo. 

In [ ]:

# Limpieza de duplicados
df = df.drop_duplicates()

muestra_df_pre_conversion = df.isnull().sum().copy()

# Saneamiento de strings
df['industry'] = df['industry'].str.lower().str.strip()

for col in ['salary_before_usd', 'salary_after_usd']:
    df[col] = df[col].str.replace(r'[^\d.]+', '', regex=True).astype(str)

df['salary_before_usd'] = pd.to_numeric(df['salary_before_usd'], errors='coerce')
df['salary_after_usd'] = pd.to_numeric(df['salary_after_usd'], errors='coerce')

muestra_df_post_conversion = df.isnull().sum().copy()

if (muestra_df_pre_conversion.equals(muestra_df_post_conversion)):
    print("Se mantiene la cantidad de nulos pre y post conversión.")
else:
    print("La conversión genero nulos.")
    

In [ ]:

df['industry'] = df['industry'].fillna('unknown')

df['automation_risk_percent'] = df['automation_risk_percent'].fillna(
    df.groupby(['job_role', 'industry'])['automation_risk_percent'].transform('mean')
)

df['salary_before_usd'] = df['salary_before_usd'].fillna(
    df.groupby(['job_role', 'industry'])['salary_before_usd'].transform('median')
)

df['salary_after_usd'] = df['salary_after_usd'].fillna(
    df.groupby(['job_role', 'industry'])['salary_after_usd'].transform('median')
)

nulos_restantes = df.isnull().sum().sum()

if nulos_restantes == 0:
    print("Limpieza completa.")
else:
    print(f"Restan {nulos_restantes} valores nulos.")

#### Añadido pre análisis: generación de copia para outliers

Se genera una copia del dataset previo al tratamiento de los outliers. El análisis de los mismos se detalla en la sección correspondiente. 

Los outliers son eliminados de la muestra.

In [ ]:
df_outliers = df.copy()

In [ ]:
for col in ['salary_before_usd', 'salary_after_usd']:
    
    Q1_grupal = df.groupby(['job_role', 'country'])[col].transform('quantile', q=0.25)
    Q3_grupal = df.groupby(['job_role', 'country'])[col].transform('quantile', q=0.75)
    
    IQR_grupal = Q3_grupal - Q1_grupal
    limite_inferior = Q1_grupal - 1.5 * IQR_grupal
    limite_superior = Q3_grupal + 1.5 * IQR_grupal
    
    df = df[(df[col] >= limite_inferior) & (df[col] <= limite_superior)]

### 4. Fase II: Análisis Estadístico y Exploratorio (EDA)
Utilizando las librerías Pandas y NumPy, resuelvan las siguientes problemáticas:
- **Volatilidad de Ingresos**: Calculen el rango y la varianza de los salarios antes y después de la IA. ¿Qué nos dice esto sobre la estabilidad del sector?
- **Brecha de Habilidades**: Determinen el promedio de `skill_gap_index` por país y encuentren dónde es más crítico el desfase formativo.
- **Detección de Outliers**: Identifiquen salarios que desafíen los rangos normales del mercado (valores negativos o extremos) y decidan su tratamiento.

#### Volatilidad de Ingresos

En este caso se agrupo por job_role y country, considerando el mismo puesto dentro del mercado local. Posteriormente se obtuvo el porcentaje de incremento o decremento entre los valores pre y post tanto de rango como varianza. Dichos valores fueron promediados para tener un indicador general del estado del mercado.

In [ ]:
grupos = df.groupby(['job_role', 'country'])

varianzas = grupos[['salary_before_usd', 'salary_after_usd']].var()
maximos = grupos[['salary_before_usd', 'salary_after_usd']].max()
minimos = grupos[['salary_before_usd', 'salary_after_usd']].min()

volatilidad_df = varianzas.rename(columns={'salary_before_usd': 'var_pre_ia','salary_after_usd': 'var_post_ia'})

volatilidad_df['rango_pre_ia'] = maximos['salary_before_usd'] - minimos['salary_before_usd']
volatilidad_df['rango_post_ia'] = maximos['salary_after_usd'] - minimos['salary_after_usd']

columnas_finales = ['rango_pre_ia', 'var_pre_ia', 'rango_post_ia', 'var_post_ia']
volatilidad_df = volatilidad_df[columnas_finales].reset_index()

display(volatilidad_df)

volatilidad_df['var_porcentual_rango'] = ((volatilidad_df['rango_post_ia'] - volatilidad_df['rango_pre_ia']) / volatilidad_df['rango_pre_ia']) * 100
volatilidad_df['var_porcentual_varianza'] = ((volatilidad_df['var_post_ia'] - volatilidad_df['var_pre_ia']) / volatilidad_df['var_pre_ia']) * 100

volatilidad_df = volatilidad_df.replace([np.inf, -np.inf], np.nan)

columnas_variacion = ['job_role', 'country', 'var_porcentual_rango', 'var_porcentual_varianza']

tabla_variacion = volatilidad_df[columnas_variacion]

print("Tabla de Variación Porcentual")
display(tabla_variacion)

promedio_var_rango = tabla_variacion['var_porcentual_rango'].mean()
promedio_var_varianza = tabla_variacion['var_porcentual_varianza'].mean()

print("\nPromedios Generales de Impacto de la IA")
print(f"Variación promedio del Rango: {promedio_var_rango:.2f}%")
print(f"Variación promedio de la Varianza: {promedio_var_varianza:.2f}%")

Sintetizado, comparando los valores pre y post IA se ve un aumento general del Rango en los sueldos y una mayor varianza. En otras palabras, la distancia entre el sueldo menor y el mayor aumento y la muestra exhibe sueldos muy dispares entre si. Dos personas pueden estar cobrando montos totalmente diferentes para el mismo puesto.

#### Brecha de Habilidades

In [ ]:
brecha_por_pais = df.groupby('country')['skill_gap_index'].mean().sort_values(ascending=False)

print("Promedio de Skill Gap Index por País")
display(brecha_por_pais)

pais_critico = brecha_por_pais.idxmax()
valor_critico = brecha_por_pais.max()

print(f"\nEl desfase formativo más crítico se encuentra en: {pais_critico} (Índice promedio: {valor_critico:.2f})")

#### Detección de outliers

Para detectar los outliers se usa IQR. Se segmenta por job_role y country para que el rango comprenda puestos similares dentro la misma economia local.

En resumen, el IQR representa el rango central de los datos obtenido a partir de restar el cuartil 25 al cuartil 75. No representa el punto medio sino la distancia entre el Q25 y Q75.

Posteriormente podemos usar dicha distancia multiplicada por 1,5 para establecer topes inferiores y superiores (conocidos como barreras de Tukey) para determinar los outliers.

Aquellos outliers detectados fueron eliminados en un paso previo. En el bloque siguiente se aprecia la detección en detalle de los mismos usando una copia de respaldo previo a dicho tratamiento.

Referencia: https://metricgate.com/docs/es/interquartile-range-outlier/

In [ ]:
for col in ['salary_before_usd', 'salary_after_usd']:

    print(f"\nAnálisis de outliers agrupados para {col}")

    Q1_grupal = df_outliers.groupby(['job_role', 'country'])[col].transform('quantile', q=0.25)
    Q3_grupal = df_outliers.groupby(['job_role', 'country'])[col].transform('quantile', q=0.75)
    
    IQR_grupal = Q3_grupal - Q1_grupal
    limite_inferior_grupal = Q1_grupal - 1.5 * IQR_grupal
    limite_superior_grupal = Q3_grupal + 1.5 * IQR_grupal

    df_outliers['IQR_grupal'] = IQR_grupal
    df_outliers['Q1_grupal'] = Q1_grupal
    df_outliers['Q3_grupal'] = Q3_grupal
    df_outliers['limite_inf_aplicado'] = limite_inferior_grupal
    df_outliers['limite_sup_aplicado'] = limite_superior_grupal

    outliers_inf = df_outliers[df_outliers[col] < df_outliers['limite_inf_aplicado']]
    outliers_sup = df_outliers[df_outliers[col] > df_outliers['limite_sup_aplicado']]

    cols_a_mostrar = ['job_id', 'job_role', 'industry', 'country', col, 'limite_inf_aplicado', 'Q1_grupal', 'IQR_grupal', 'Q3_grupal', 'limite_sup_aplicado']

    print(f"Sueldos atípicamente bajos: {len(outliers_inf)}")
    if len(outliers_inf) > 0:
        display(outliers_inf[cols_a_mostrar])

    print(f"Sueldos atípicamente altos: {len(outliers_sup)}")
    if len(outliers_sup) > 0:
        display(outliers_sup[cols_a_mostrar])

    df_outliers = df_outliers.drop(columns=['limite_inf_aplicado', 'limite_sup_aplicado', 'IQR_grupal', 'Q1_grupal', 'Q3_grupal'])

### 5. Fase III: Visualización y Comunicación
Diseñen las siguientes piezas utilizando Matplotlib:
- **Gráfico de Evolución**: Tendencia anual del salario promedio en finanzas desde 2020 a 2026.
- **Gráfico de Distribución (Pie/Pie Chart)**: Distribución de los roles según su categoría de riesgo de automatización.
- **Heatmap o Barras Geográficas**: Comparativa del `ai_adoption_level` por país para este sector.

### 6. Fase IV: Reflexión Crítica (Anti-LLM)
*Responda basándose estrictamente en los hallazgos de su procesamiento de datos:*
- ¿Existe evidencia de que la IA esté "empobreciendo" los roles tradicionales de finanzas?
- Según sus datos, ¿la factibilidad de trabajo remoto protege o expone más al trabajador financiero frente al reemplazo por IA?

### 7. Requerimientos de Entrega y Defensa
1.  **Informe Profesional**: PDF o Markdown con resumen ejecutivo, bitácora de saneamiento e interpretación de resultados.
2.  **Código Fuente**: Script o Notebook estructurado, comentado y reproducible.
3.  **Defensa Oral**: Justificación técnica de la limpieza e interpretación de las visualizaciones ante el curso.
